# CENG 467 — Question 2: Named Entity Recognition

**Author:** Yusuf Furkan Aktay  
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Instructor:** Prof. Dr. Aytuğ Onan

## Objective
Examine sequence labeling and contextual modeling in named entity recognition by comparing a hybrid BiLSTM-CRF model against a transformer-based DistilBERT model on the CoNLL-2003 English dataset.

## Setup
- **Runtime:** Google Colab — GPU (T4)
- **Random Seed:** 42
- **Dataset:** CoNLL-2003 (English, BIO-tagged)
- **Metrics:** Precision, Recall, F1 (entity-level via `seqeval`)

Run `Runtime → Run all` after selecting GPU.

## 1. Environment Setup

In [1]:
!pip install -q --upgrade transformers accelerate datasets
!pip install -q seqeval pytorch-crf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os, random, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
os.makedirs('results', exist_ok=True)

Device: cuda


## 2. Dataset: CoNLL-2003

CoNLL-2003 is the canonical English NER benchmark, derived from Reuters news articles. It contains four entity types — **PER** (person), **LOC** (location), **ORG** (organization), and **MISC** (miscellaneous) — annotated with the BIO tagging scheme (B = beginning, I = inside, O = outside).
Splits: 14,041 train / 3,250 dev / 3,453 test sentences.

In [3]:
from datasets import load_dataset

# Parquet-based mirror — script kullanmıyor, datasets'in yeni versiyonlarıyla uyumlu
ds = load_dataset('tomaarsen/conll2003')

# Etiket listesini manuel tanımla (CoNLL-2003 standart BIO etiketleri)
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in id2label.items()}

print('Labels:', label_list)
print(f'Train: {len(ds["train"])} | Validation: {len(ds["validation"])} | Test: {len(ds["test"])}')
print('Sample tokens:', ds['train'][0]['tokens'][:10])
print('Sample tags  :', [id2label[i] for i in ds['train'][0]['ner_tags'][:10]])

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/316k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/288k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 14041 | Validation: 3250 | Test: 3453
Sample tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Sample tags  : ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


## 3. Model 1 — BiLSTM-CRF (Classical/Hybrid)

We build a vocab from the training set, embed tokens, run a BiLSTM, and decode with a CRF layer for proper BIO-consistency.

In [4]:
from collections import Counter
from torchcrf import CRF

PAD_ID, UNK_ID = 0, 1
MAX_LEN = 80

# Vocab from training tokens (lowercased)
counter = Counter()
for ex in ds['train']:
    counter.update([t.lower() for t in ex['tokens']])
vocab = {'<pad>': PAD_ID, '<unk>': UNK_ID}
for w, c in counter.most_common(30000):
    if c >= 2: vocab[w] = len(vocab)
VOCAB_SIZE = len(vocab); NUM_TAGS = len(label_list)
PAD_TAG = label2id['O']
print(f'Vocab: {VOCAB_SIZE} | Tags: {NUM_TAGS}')

def encode_lstm(example):
    toks = [vocab.get(t.lower(), UNK_ID) for t in example['tokens']][:MAX_LEN]
    tags = list(example['ner_tags'])[:MAX_LEN]
    mask = [1]*len(toks) + [0]*(MAX_LEN-len(toks))
    toks += [PAD_ID]*(MAX_LEN-len(toks))
    tags += [PAD_TAG]*(MAX_LEN-len(tags))
    return toks, tags, mask

class ConllLSTMDS(Dataset):
    def __init__(self, hf_split):
        self.x, self.y, self.m = [], [], []
        for ex in hf_split:
            t, g, mk = encode_lstm(ex)
            self.x.append(t); self.y.append(g); self.m.append(mk)
        self.x = torch.tensor(self.x, dtype=torch.long)
        self.y = torch.tensor(self.y, dtype=torch.long)
        self.m = torch.tensor(self.m, dtype=torch.bool)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], self.m[i]

train_dl = DataLoader(ConllLSTMDS(ds['train']),      batch_size=32, shuffle=True)
test_dl  = DataLoader(ConllLSTMDS(ds['test']),       batch_size=64)
test_raw_tokens = [ex['tokens']   for ex in ds['test']]
test_raw_tags   = [[id2label[t] for t in ex['ner_tags']] for ex in ds['test']]

Vocab: 10951 | Tags: 9


In [5]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, num_tags, emb_dim=100, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True, num_layers=1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden*2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
    def emissions(self, x):
        e = self.emb(x); o,_ = self.lstm(e); return self.fc(self.drop(o))
    def loss(self, x, tags, mask):
        em = self.emissions(x)
        return -self.crf(em, tags, mask=mask, reduction='mean')
    def decode(self, x, mask):
        em = self.emissions(x)
        return self.crf.decode(em, mask=mask)

model_lstm = BiLSTMCRF(VOCAB_SIZE, NUM_TAGS).to(DEVICE)
opt = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)

EPOCHS_LSTM = 5
t0 = time.time()
for epoch in range(EPOCHS_LSTM):
    model_lstm.train(); total = 0; n = 0
    for x, y, m in train_dl:
        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        opt.zero_grad()
        loss = model_lstm.loss(x, y, m)
        loss.backward(); opt.step()
        total += loss.item(); n += 1
    print(f'Epoch {epoch+1}: loss={total/n:.4f}')
lstm_time = time.time() - t0

# Decode predictions on test set
model_lstm.eval(); lstm_pred_tags = []
with torch.no_grad():
    for x, y, m in test_dl:
        x, m = x.to(DEVICE), m.to(DEVICE)
        decoded = model_lstm.decode(x, m)
        for seq in decoded:
            lstm_pred_tags.append([id2label[t] for t in seq])

# Truncate to original sentence length
lstm_pred_tags_aligned = [seq[:len(test_raw_tags[i])] for i, seq in enumerate(lstm_pred_tags)]
lstm_true_tags_aligned = [seq[:len(lstm_pred_tags_aligned[i])] for i, seq in enumerate(test_raw_tags)]
print(f'BiLSTM-CRF train time: {lstm_time:.1f}s')

Epoch 1: loss=7.9887
Epoch 2: loss=3.6543
Epoch 3: loss=2.2439
Epoch 4: loss=1.5032
Epoch 5: loss=1.0410
BiLSTM-CRF train time: 188.8s


In [6]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report as seq_report

lstm_p = precision_score(lstm_true_tags_aligned, lstm_pred_tags_aligned)
lstm_r = recall_score(lstm_true_tags_aligned, lstm_pred_tags_aligned)
lstm_f = f1_score(lstm_true_tags_aligned, lstm_pred_tags_aligned)
print(f'BiLSTM-CRF | P={lstm_p:.4f}  R={lstm_r:.4f}  F1={lstm_f:.4f}')
print('\n--- Per-entity breakdown ---')
print(seq_report(lstm_true_tags_aligned, lstm_pred_tags_aligned, digits=4))

BiLSTM-CRF | P=0.7606  R=0.6587  F1=0.7060

--- Per-entity breakdown ---
              precision    recall  f1-score   support

         LOC     0.8485    0.7444    0.7931      1663
        MISC     0.7519    0.5783    0.6538       702
         ORG     0.7080    0.5780    0.6364      1661
         PER     0.7264    0.6886    0.7070      1612

   micro avg     0.7606    0.6587    0.7060      5638
   macro avg     0.7587    0.6473    0.6976      5638
weighted avg     0.7602    0.6587    0.7050      5638



## 4. Model 2 — DistilBERT-NER (Transformer)

We fine-tune `distilbert-base-uncased` with a token-classification head. Subword alignment uses the standard "only label the first subword" convention; subsequent subwords get label `-100` (ignored by the loss).

In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.optim import AdamW

MODEL_NAME = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN_BERT = 128
IGNORE_IDX = -100

def encode_bert(tokens, tags):
    enc = tok(tokens, is_split_into_words=True, truncation=True,
              padding='max_length', max_length=MAX_LEN_BERT, return_tensors='pt')
    word_ids = enc.word_ids(0)
    labels = []
    prev = None
    for wid in word_ids:
        if wid is None: labels.append(IGNORE_IDX)
        elif wid != prev: labels.append(tags[wid])
        else: labels.append(IGNORE_IDX)
        prev = wid
    return enc['input_ids'][0], enc['attention_mask'][0], torch.tensor(labels), word_ids

class ConllBertDS(Dataset):
    def __init__(self, hf_split):
        self.items = []
        for ex in hf_split:
            ids, mask, lbl, wids = encode_bert(ex['tokens'], ex['ner_tags'])
            self.items.append((ids, mask, lbl, wids, ex['tokens'], ex['ner_tags']))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ids, mask, lbl, _, _, _ = self.items[i]
        return ids, mask, lbl

bert_train_ds = ConllBertDS(ds['train'])
bert_test_ds  = ConllBertDS(ds['test'])
bert_train_dl = DataLoader(bert_train_ds, batch_size=16, shuffle=True)
bert_test_dl  = DataLoader(bert_test_ds,  batch_size=32)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
from tqdm.auto import tqdm

# Hızlandırma: 5000 örneklik subset (T4 Free için optimize)
# Tam veri seti yerine bu küçük subset ~3-4 dakikada biter, F1 sadece ~1-2 puan düşer.
small_train = ds['train'].select(range(5000))
bert_train_ds_small = ConllBertDS(small_train)
bert_train_dl_small = DataLoader(bert_train_ds_small, batch_size=16, shuffle=True)

model_bert = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_TAGS, id2label=id2label, label2id=label2id
).to(DEVICE)
optimizer = AdamW(model_bert.parameters(), lr=3e-5, weight_decay=0.01)

EPOCHS_BERT = 2
t0 = time.time()
for epoch in range(EPOCHS_BERT):
    model_bert.train(); total = 0; n = 0
    pbar = tqdm(bert_train_dl_small, desc=f'Epoch {epoch+1}/{EPOCHS_BERT}')
    for ids, mask, lbl in pbar:
        ids, mask, lbl = ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        out = model_bert(input_ids=ids, attention_mask=mask, labels=lbl)
        out.loss.backward()
        optimizer.step()
        total += out.loss.item(); n += 1
        pbar.set_postfix({'loss': f'{total/n:.4f}'})
    print(f'Epoch {epoch+1} done: avg_loss={total/n:.4f}')
bert_time = time.time() - t0
print(f'\nDistilBERT-NER train time: {bert_time:.1f}s')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/2:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 1 done: avg_loss=0.2483


Epoch 2/2:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 2 done: avg_loss=0.0443

DistilBERT-NER train time: 112.9s


In [9]:
model_bert = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_TAGS, id2label=id2label, label2id=label2id
).to(DEVICE)
optimizer = AdamW(model_bert.parameters(), lr=3e-5, weight_decay=0.01)

EPOCHS_BERT = 2
t0 = time.time()
for epoch in range(EPOCHS_BERT):
    model_bert.train(); total = 0; n = 0
    for ids, mask, lbl in bert_train_dl:
        ids, mask, lbl = ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        out = model_bert(input_ids=ids, attention_mask=mask, labels=lbl)
        out.loss.backward(); optimizer.step()
        total += out.loss.item(); n += 1
    print(f'Epoch {epoch+1}: loss={total/n:.4f}')
bert_time = time.time() - t0
print(f'BERT-NER train time: {bert_time:.1f}s')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1: loss=0.1340
Epoch 2: loss=0.0356
BERT-NER train time: 317.8s


## 5. Final Results & Error Analysis

In [16]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report as seq_report

# Inference + word-level alignment
model_bert.eval()
bert_pred_tags, bert_true_tags = [], []
with torch.no_grad():
    for i, (ids, mask, lbl, wids, raw_toks, raw_tags) in enumerate(bert_test_ds.items):
        logits = model_bert(input_ids=ids.unsqueeze(0).to(DEVICE),
                            attention_mask=mask.unsqueeze(0).to(DEVICE)).logits[0]
        preds_subword = logits.argmax(-1).cpu().tolist()
        word_preds = []
        prev_wid = None
        for j, wid in enumerate(wids):
            if wid is None or wid == prev_wid:
                continue
            word_preds.append(id2label[preds_subword[j]])
            prev_wid = wid
        word_preds = word_preds[:len(raw_toks)]
        if len(word_preds) < len(raw_toks):
            word_preds += ['O'] * (len(raw_toks) - len(word_preds))
        bert_pred_tags.append(word_preds)
        bert_true_tags.append([id2label[t] for t in raw_tags])

bert_p = precision_score(bert_true_tags, bert_pred_tags)
bert_r = recall_score(bert_true_tags, bert_pred_tags)
bert_f = f1_score(bert_true_tags, bert_pred_tags)
print(f'DistilBERT-NER | P={bert_p:.4f}  R={bert_r:.4f}  F1={bert_f:.4f}')
print('\n--- Per-entity breakdown ---')
print(seq_report(bert_true_tags, bert_pred_tags, digits=4))

DistilBERT-NER | P=0.8916  R=0.9014  F1=0.8965

--- Per-entity breakdown ---
              precision    recall  f1-score   support

         LOC     0.8912    0.9335    0.9119      1668
        MISC     0.7868    0.7991    0.7929       702
         ORG     0.8555    0.8627    0.8591      1661
         PER     0.9778    0.9524    0.9649      1617

   micro avg     0.8916    0.9014    0.8965      5648
   macro avg     0.8778    0.8869    0.8822      5648
weighted avg     0.8925    0.9014    0.8968      5648



In [17]:
import pandas as pd

summary = pd.DataFrame([
    {'Model':'BiLSTM-CRF',     'Precision':lstm_p,'Recall':lstm_r,'F1':lstm_f,'Train Time (s)':lstm_time},
    {'Model':'DistilBERT-NER', 'Precision':bert_p,'Recall':bert_r,'F1':bert_f,'Train Time (s)':bert_time},
])
print(summary.to_string(index=False))
summary.to_csv('results/q2_summary.csv', index=False)

         Model  Precision   Recall       F1  Train Time (s)
    BiLSTM-CRF   0.760598 0.658744 0.706017      188.759508
DistilBERT-NER   0.891594 0.901381 0.896461      317.757779


In [18]:
import json

# Error analysis: BERT'in yanlış etiketlediği cümleleri bul
errors = []
for i in range(len(test_raw_tokens)):
    true = bert_true_tags[i]
    pred_bert = bert_pred_tags[i]
    pred_lstm = lstm_pred_tags_aligned[i] if i < len(lstm_pred_tags_aligned) else ['O']*len(true)
    pred_lstm = pred_lstm[:len(true)] + ['O']*(len(true)-len(pred_lstm))
    bert_wrong = [j for j in range(len(true)) if true[j] != pred_bert[j]]
    if bert_wrong:
        errors.append({
            'tokens': test_raw_tokens[i],
            'true':   true,
            'pred_bert': pred_bert,
            'pred_lstm': pred_lstm,
            'wrong_idx': bert_wrong,
        })

print(f'BERT made errors in {len(errors)} / {len(bert_true_tags)} sentences\n')
print('--- 5 sample error sentences (BERT mistakes) ---')
for k, e in enumerate(errors[:5]):
    print(f'\nSentence {k+1}:')
    for j, tok_w in enumerate(e['tokens']):
        marker = ' <-- ERR' if j in e['wrong_idx'] else ''
        print(f"  {tok_w:<20} true={e['true'][j]:<8} bert={e['pred_bert'][j]:<8} lstm={e['pred_lstm'][j]:<8}{marker}")

with open('results/q2_errors.json','w') as f:
    json.dump(errors[:20], f, indent=2)
with open('results/q2_full.json','w') as f:
    json.dump({'bilstm_crf':{'precision':lstm_p,'recall':lstm_r,'f1':lstm_f,'time_sec':lstm_time},
               'distilbert_ner':{'precision':bert_p,'recall':bert_r,'f1':bert_f,'time_sec':bert_time}}, f, indent=2)
print('\nResults saved to results/')

BERT made errors in 582 / 3453 sentences

--- 5 sample error sentences (BERT mistakes) ---

Sentence 1:
  SOCCER               true=O        bert=O        lstm=O       
  -                    true=O        bert=O        lstm=O       
  JAPAN                true=B-LOC    bert=B-LOC    lstm=B-LOC   
  GET                  true=O        bert=O        lstm=O       
  LUCKY                true=O        bert=O        lstm=O       
  WIN                  true=O        bert=O        lstm=O       
  ,                    true=O        bert=O        lstm=O       
  CHINA                true=B-PER    bert=B-LOC    lstm=B-LOC    <-- ERR
  IN                   true=O        bert=O        lstm=O       
  SURPRISE             true=O        bert=O        lstm=O       
  DEFEAT               true=O        bert=O        lstm=O       
  .                    true=O        bert=O        lstm=O       

Sentence 2:
  Cuttitta             true=B-PER    bert=B-PER    lstm=B-PER   
  announced            true=O 

## 6. Discussion (in Report)

See `reports/CENG467_Midterm_Report.pdf` for analysis of:
- Boundary errors (B-/I- confusions)
- Entity-type confusion (ORG↔LOC, PER↔ORG)
- The role of contextual embeddings in disambiguating polysemous tokens
- BiLSTM-CRF vs Transformer trade-offs